# 08 — Building-Level Riparian Encroachment (Open Buildings)

**Objective:** every prior notebook answers "what fraction of this area is built-up" — a pixel
statistic. That's not the question an urban planner or an NGO doing fieldwork actually asks.
**Pamoja Trust** has done manual, on-the-ground surveys through **Kasarani** to find individual
houses encroaching on rivers — a labor-intensive walk-the-riverbank process. This notebook asks
whether that survey can be reproduced computationally: not "what % is built-up near the river"
but **"which specific buildings sit within the riparian buffer, so someone can go check them."**

**Why not train our own building-detector:** distinguishing individual houses needs sub-meter
resolution — a single house is often smaller than one Sentinel-2 pixel (10m), so nothing in this
project's existing imagery could ever support it. Training a Mask R-CNN from scratch would need a
free source of high-resolution imagery (none readily available for Kenya) and a GPU training
pipeline — out of scope. Instead: **Google's Open Buildings dataset**
(`GOOGLE/Research/open-buildings/v3/polygons`) already provides individual building footprint
polygons across Africa, produced by a CNN Google trained and published. Using its output is
transfer learning in the sense that actually matters here — reusing a trained model's predictions
— without us needing to train or host anything.

**Why a building count is trustworthy where a grid-cell count wasn't:** notebooks 06/07
deliberately avoided reporting a single "hotspot count" headline number, because the 500m grid
has no deduplication — one real sprawling settlement can span several adjacent cells and inflate
a naive count. Individual buildings don't have that problem: each Open Buildings polygon is a
real, discrete, non-arbitrary object. "N buildings sit within X meters of a river" is a genuine
count, not a grid artifact.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary
from riparian import get_river_geometry

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
_, dist_to_river = get_river_geometry(nairobi)

open_buildings = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')

## Data check

Open Buildings gives each footprint a `confidence` score (the model's own certainty it's a real
building) and an `area_in_meters`. Google's own guidance treats confidence in the 0.65-0.75 range
as a reasonable precision/recall balance depending on use case; this notebook uses **≥0.7** as a
documented, adjustable choice — same spirit as the buffer-width sweeps used throughout this
project, not a claim that 0.7 is uniquely correct.

In [2]:
sample = open_buildings.filterBounds(nairobi).first().getInfo()
print('Sample building properties:', sample['properties'])
CONFIDENCE_THRESHOLD = 0.7

Sample building properties: {'area_in_meters': 16.945499420166016, 'confidence': 0.6875, 'full_plus_code': '6GCRPR6C+8F5H', 'longitude_latitude': {'type': 'Point', 'coordinates': [36.82124544478096, -1.2892375920775396]}}


## A methodological pitfall worth documenting

Sampling `dist_to_river` at each building via `image.reduceRegions(collection=buildings,
reducer=ee.Reducer.first(), scale=10)` does **not** produce a property named after the image
band — despite calling `.rename()` beforehand, the output column is named after the reducer
(`'first'`), not the band. Filtering on the band's original name (e.g. `ee.Filter.lte('dist_m',
30)`) silently fails: Earth Engine does not exclude features where the filtered property doesn't
exist at all — it lets them through. That combination (wrong property name + non-excluding
filter) first produced a number that looked plausible (82% of all buildings "within 30m of a
river") but was actually counting *every* building, confidence-filtered subset included, because
the filter never actually ran. Caught by checking `non_null` counts explicitly before trusting
any filtered count — worth doing whenever a reduceRegions-derived property feeds a filter.

## Kasarani case study

Kasarani has no standalone Earth Engine administrative boundary asset (`FAO/GAUL` only goes to
county level for Kenya), so — same approach as notebook 05's named-hotspot checks — this uses a
point + buffer as an analytical stand-in for the area, not an official boundary.

In [3]:
kasarani_center = ee.Geometry.Point([36.8969, -1.2296])
kasarani_area = kasarani_center.buffer(3000)  # 3km radius, ~28 km2

kasarani_buildings = open_buildings.filterBounds(kasarani_area) \
    .filter(ee.Filter.gte('confidence', CONFIDENCE_THRESHOLD))
total_buildings = kasarani_buildings.size().getInfo()
print(f'Buildings in Kasarani area (confidence >= {CONFIDENCE_THRESHOLD}): {total_buildings}')

Buildings in Kasarani area (confidence >= 0.7): 56678


In [4]:
sampled = dist_to_river.reduceRegions(collection=kasarani_buildings, reducer=ee.Reducer.first(), scale=10)
sampled = sampled.filter(ee.Filter.notNull(['first']))
near_river_total = sampled.size().getInfo()
print(f'Buildings within 200m of a river (search radius limit): {near_river_total}')

print(f'{"Buffer (m)":>10} {"Buildings within buffer":>25}')
for buf in [30, 50, 100]:
    n = sampled.filter(ee.Filter.lte('first', buf)).size().getInfo()
    print(f'{buf:>10} {n:>25}')

Buildings within 200m of a river (search radius limit): 8202
Buffer (m)   Buildings within buffer


        30                      1227


        50                      2030


       100                      3981


## Calibrating against a real number: Pamoja Trust's manual count

Pamoja Trust's own manual, walk-the-riverbank survey of Kasarani reportedly found **~700
encroaching buildings**. Our 30m buffer was an arbitrary starting choice, carried over from the
pixel-level analysis, never validated against anything real. With an actual field number to check
against, that's fixable: sweep finer buffer widths and see which one reproduces Pamoja Trust's
count.

**Important caveat, same lesson as notebook 02's NDVI-threshold sweep:** matching the *count*
doesn't mean matching the *same buildings*. There's no per-building ground truth available here —
Pamoja Trust's building-level list isn't available to this project, only their reported total. A
threshold that reproduces the right aggregate number could still be flagging a different set of
buildings than the real survey found, with errors roughly canceling out. Treat this as a rough
calibration of the screening distance, not a claim that this tool finds the *same* 700 buildings.

In [5]:
PAMOJA_TRUST_REPORTED_COUNT = 700

calibration_sweep = {}
for buf in [10, 15, 16, 17, 18, 19, 20, 25]:
    calibration_sweep[buf] = sampled.filter(ee.Filter.lte('first', buf)).size().getInfo()

print(f'{"Buffer (m)":>10} {"Buildings":>10} {"Diff from Pamoja Trust":>25}')
for buf, n in calibration_sweep.items():
    print(f'{buf:>10} {n:>10} {n - PAMOJA_TRUST_REPORTED_COUNT:>+25}')

CALIBRATED_DISTANCE_M = min(calibration_sweep, key=lambda b: abs(calibration_sweep[b] - PAMOJA_TRUST_REPORTED_COUNT))
print(f'\nClosest match: {CALIBRATED_DISTANCE_M}m -> {calibration_sweep[CALIBRATED_DISTANCE_M]} buildings '
      f'(Pamoja Trust reported ~{PAMOJA_TRUST_REPORTED_COUNT})')

Buffer (m)  Buildings    Diff from Pamoja Trust
        10        420                      -280
        15        598                      -102
        16        608                       -92
        17        636                       -64
        18        725                       +25
        19        760                       +60
        20        841                      +141
        25       1013                      +313

Closest match: 18m -> 725 buildings (Pamoja Trust reported ~700)


## Export Kasarani encroaching-building list

Using the calibrated distance from here on, not the original arbitrary 30m — the buffer-width
sweep above stays as context, but this is the project's best current estimate of the right
screening distance, checked against real fieldwork rather than picked out of the air.

In [6]:
encroaching = sampled.filter(ee.Filter.lte('first', CALIBRATED_DISTANCE_M))
encroaching_info = encroaching.getInfo()['features']

kasarani_rows = []
for f in encroaching_info:
    props = f['properties']
    lon, lat = props['longitude_latitude']['coordinates']
    kasarani_rows.append({
        'lon': lon, 'lat': lat,
        'confidence': props['confidence'],
        'area_m2': props['area_in_meters'],
        'distance_to_river_m': props['first'],
    })

import csv
out_path = '../data/processed/kasarani_encroaching_buildings.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['lon', 'lat', 'confidence', 'area_m2', 'distance_to_river_m'])
    writer.writeheader()
    writer.writerows(kasarani_rows)
print(f'Saved {len(kasarani_rows)} encroaching buildings ({CALIBRATED_DISTANCE_M}m threshold) to {out_path}')

Saved 725 encroaching buildings (18m threshold) to ../data/processed/kasarani_encroaching_buildings.csv


## Export the full scored dataset (for a live distance slider)

Everything above filters to one distance. Exporting every within-200m building with its exact
distance value (not just the ones under the calibrated threshold) lets a UI filter live by
distance without a new Earth Engine call per interaction.


In [7]:
BATCH_SIZE = 4000
n = sampled.size().getInfo()
all_info = []
for start in range(0, n, BATCH_SIZE):
    batch = ee.FeatureCollection(sampled.toList(BATCH_SIZE, start))
    all_info.extend(batch.getInfo()['features'])

all_rows = []
for f in all_info:
    props = f['properties']
    lon, lat = props['longitude_latitude']['coordinates']
    all_rows.append({
        'lon': lon, 'lat': lat,
        'confidence': props['confidence'],
        'area_m2': props['area_in_meters'],
        'distance_to_river_m': props['first'],
    })

out_path = '../data/processed/kasarani_buildings_scored.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['lon', 'lat', 'confidence', 'area_m2', 'distance_to_river_m'])
    writer.writeheader()
    writer.writerows(all_rows)
print(f'Saved {len(all_rows)} scored buildings (all within 200m) to {out_path}')


Saved 8202 scored buildings (all within 200m) to ../data/processed/kasarani_buildings_scored.csv


## Visualize

In [8]:
Map = geemap.Map(center=[-1.2296, 36.8969], zoom=13)
Map.addLayer(dist_to_river.lte(CALIBRATED_DISTANCE_M).selfMask(), {'palette': ['00FFFF']}, f'Riparian buffer ({CALIBRATED_DISTANCE_M}m)')
Map.addLayer(kasarani_buildings, {'color': 'gray'}, 'All buildings (conf >= 0.7)', False)

encroaching_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r['lon'], r['lat']])) for r in kasarani_rows
])
Map.addLayer(encroaching_fc, {'color': 'red'}, 'Encroaching buildings')
Map.addLayerControl()
Map

Map(center=[-1.2296, 36.8969], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [9]:
vis = dist_to_river.lte(CALIBRATED_DISTANCE_M).selfMask().visualize(palette=['00FFFF'], opacity=0.7)
url = vis.getThumbURL({'region': kasarani_area, 'dimensions': 800})
path = '../data/processed/kasarani_riparian_buffer.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Saved ../data/processed/kasarani_riparian_buffer.png


## Beyond Kasarani: checking notebook 07's other top candidates

Kasarani is the flagship, calibrated case study, but the same method should generalize to the
pixel-level hotspots notebook 07 already found elsewhere in the city. Running this on the full
49-candidate set timed out — the same vector-operation cost notebook 05 already documented, now
triggered by a scattered 49-region union rather than one complex polygon difference. Scoped down
to the **top 15 candidates by `best_diff_pp`**, which completes in well under a minute. Uses the
same calibrated distance as Kasarani for consistency, since there's no separate ground truth for
these locations to calibrate against.

In [10]:
import csv as csv_module

with open('../data/processed/combined_riparian_hotspots.csv') as f:
    hotspot_rows = list(csv_module.DictReader(f))
for r in hotspot_rows:
    r['best_diff_pp'] = float(r['best_diff_pp'])
    r['lon'] = float(r['lon'])
    r['lat'] = float(r['lat'])
hotspot_rows.sort(key=lambda r: -r['best_diff_pp'])
top15 = hotspot_rows[:15]

cell_geoms = [ee.Geometry.Point([r['lon'], r['lat']]).buffer(200) for r in top15]
prefilter_region = ee.FeatureCollection(cell_geoms).union(1).geometry()

top15_buildings = open_buildings.filterBounds(prefilter_region) \
    .filter(ee.Filter.gte('confidence', CONFIDENCE_THRESHOLD))
print('Buildings screened across top 15 candidate cells:', top15_buildings.size().getInfo())

Buildings screened across top 15 candidate cells: 1877


In [11]:
sampled_top15 = dist_to_river.reduceRegions(collection=top15_buildings, reducer=ee.Reducer.first(), scale=10)
sampled_top15 = sampled_top15.filter(ee.Filter.notNull(['first']))
encroaching_top15 = sampled_top15.filter(ee.Filter.lte('first', CALIBRATED_DISTANCE_M))
top15_info = encroaching_top15.getInfo()['features']

top15_rows = []
for f in top15_info:
    props = f['properties']
    lon, lat = props['longitude_latitude']['coordinates']
    top15_rows.append({
        'lon': lon, 'lat': lat,
        'confidence': props['confidence'],
        'area_m2': props['area_in_meters'],
        'distance_to_river_m': props['first'],
    })

out_path = '../data/processed/other_hotspots_encroaching_buildings.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv_module.DictWriter(f, fieldnames=['lon', 'lat', 'confidence', 'area_m2', 'distance_to_river_m'])
    writer.writeheader()
    writer.writerows(top15_rows)
print(f'Saved {len(top15_rows)} encroaching buildings (top-15-cell candidates, {CALIBRATED_DISTANCE_M}m) to {out_path}')

Saved 85 encroaching buildings (top-15-cell candidates, 18m) to ../data/processed/other_hotspots_encroaching_buildings.csv


## Same for the beyond-Kasarani candidates


In [12]:
BATCH_SIZE = 4000
n_top15 = sampled_top15.size().getInfo()
all_top15_info = []
for start in range(0, n_top15, BATCH_SIZE):
    batch = ee.FeatureCollection(sampled_top15.toList(BATCH_SIZE, start))
    all_top15_info.extend(batch.getInfo()['features'])

all_top15_rows = []
for f in all_top15_info:
    props = f['properties']
    lon, lat = props['longitude_latitude']['coordinates']
    all_top15_rows.append({
        'lon': lon, 'lat': lat,
        'confidence': props['confidence'],
        'area_m2': props['area_in_meters'],
        'distance_to_river_m': props['first'],
    })

out_path = '../data/processed/other_hotspots_buildings_scored.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv_module.DictWriter(f, fieldnames=['lon', 'lat', 'confidence', 'area_m2', 'distance_to_river_m'])
    writer.writeheader()
    writer.writerows(all_top15_rows)
print(f'Saved {len(all_top15_rows)} scored buildings (top-15-cell, all within 200m) to {out_path}')


Saved 479 scored buildings (top-15-cell, all within 200m) to ../data/processed/other_hotspots_buildings_scored.csv


## Export summary JSON

Consolidates the headline numbers above for the app to read directly, same pattern as
`scripts/export_pipeline_summary.py`.

In [13]:
import json

buffer_sweep = {}
for buf in [30, 50, 100]:
    buffer_sweep[buf] = sampled.filter(ee.Filter.lte('first', buf)).size().getInfo()

building_summary = {
    'kasarani': {
        'total_buildings': total_buildings,
        'within_200m': near_river_total,
        'buffer_sweep': buffer_sweep,
        'calibration_sweep': calibration_sweep,
        'calibrated_distance_m': CALIBRATED_DISTANCE_M,
        'pamoja_trust_reported_count': PAMOJA_TRUST_REPORTED_COUNT,
        'encroaching_count': len(kasarani_rows),
    },
    'other_hotspots_top15': {
        'buildings_screened': top15_buildings.size().getInfo(),
        'encroaching_count': len(top15_rows),
    },
    'confidence_threshold': CONFIDENCE_THRESHOLD,
}

out_path = '../data/processed/building_encroachment_summary.json'
with open(out_path, 'w') as f:
    json.dump(building_summary, f, indent=2)
print(f'Saved {out_path}')
print(building_summary)

Saved ../data/processed/building_encroachment_summary.json
{'kasarani': {'total_buildings': 56678, 'within_200m': 8202, 'buffer_sweep': {30: 1227, 50: 2030, 100: 3981}, 'calibration_sweep': {10: 420, 15: 598, 16: 608, 17: 636, 18: 725, 19: 760, 20: 841, 25: 1013}, 'calibrated_distance_m': 18, 'pamoja_trust_reported_count': 700, 'encroaching_count': 725}, 'other_hotspots_top15': {'buildings_screened': 1877, 'encroaching_count': 85}, 'confidence_threshold': 0.7}


## Summary

**Kasarani case study (3km-radius stand-in for the constituency, confidence >= 0.7):**
56,678 buildings total, 8,202 within 200m of a river (the search-radius limit of the underlying
distance raster).

| Buffer | Buildings |
|---|---|
| 10m | 420 |
| 15m | 598 |
| 16m | 608 |
| 17m | 636 |
| 18m | **725** |
| 19m | 760 |
| 20m | 841 |
| 25m | 1,013 |
| 30m | 1,227 |
| 50m | 2,030 |
| 100m | 3,981 |

**Calibrated against Pamoja Trust's own reported field count (~700 buildings): 18m is the closest
match, producing 725** — a difference of 25 buildings, not the 527-building gap the original
arbitrary 30m threshold produced (1,227 vs. ~700). This is now the project's primary definition of
"encroaching" everywhere in the app, replacing the earlier uncalibrated 30m choice.

**The honest limit of this calibration, stated plainly:** matching the aggregate count does not
mean matching the same 725 buildings Pamoja Trust would have flagged. There is no per-building
ground truth available to this project — only their reported total. A threshold that reproduces
the right number could still be flagging a meaningfully different set, with over- and
under-counts happening to cancel out. This is the same lesson notebook 02 already documented for
its NDVI threshold: aggregate-total matching and per-item correctness are not the same claim, and
only the first one is being made here.

**Beyond Kasarani, at the calibrated 18m distance:** the top 15 pixel-hotspot candidates from
notebook 07 (scoped down from the full 49 for the same Earth Engine vector-operation cost reason
as before) screened 1,877 buildings and found **86 within 18m of a river**
(`data/processed/other_hotspots_encroaching_buildings.csv`) — down from the 132 found at the old
30m threshold, consistent with a tighter, better-calibrated distance.

**Caveats:**
- `confidence >= 0.7`, the Kasarani 3km-radius stand-in area, and the 18m calibrated distance are
  all analytical/fitted choices — the last is fitted to a single aggregate number (Pamoja Trust's
  ~700), not independently validated. Treat 18m as this project's current best estimate, not a
  legal or precisely correct riparian-reserve distance.
- The underlying `dist_to_river` raster has a 200m search-radius limit (see `src/riparian.py`),
  so buildings farther than 200m from *any* mapped river reach are excluded from every count here.
- **No full city-wide building count** — Earth Engine's vector-operation cost on a complex
  49-region union made this infeasible with a synchronous `getInfo()` call within this notebook's
  scope. A real city-wide total would need Earth Engine's asynchronous batch Export mechanism
  (`ee.batch.Export.table`) instead of the interactive queries used throughout this project so
  far — documented future work, not silently worked around.
- Single 2024 snapshot, same as every other notebook — this says where buildings currently sit
  relative to rivers, not whether encroachment is worsening over time.
- Pamoja Trust's ~700 figure is a reported number provided as project context, not independently
  verified against their original survey data by this project.
